# 因子评估

## 一、导入相关库

In [2]:
import os
import sys
import warnings
import inspect
import re
import time
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from datetime import datetime, date, timedelta
# 忽视警告信息
warnings.filterwarnings(action='ignore')

# 进度条和并行处理
from tqdm import tqdm
from joblib import delayed, Parallel

# alphalens相关
from alphalens.utils import *

# 因子评估相关
from utils import factor_evaluator as fe
from utils import factor_plotting as fp

# 因子脚本
import factor_script.factor_calendar_2025 as fc2025
import factor_script.factor_njy as fcnjy

# Jupyter显示相关
from IPython.display import Markdown

# 导入数据接口sdk
import zenidatasdk as zd
client = zd.Client()

print(f"zenidatasdk当前版本: {zd.__version__}")

zenidatasdk当前版本: 2.0.8


## 二、获取数据

### 指定回测范围，指数

In [3]:
# 最新日
yesterday = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

# 历史回测区间
init_date = '2016-01-01'
start_date = '2018-01-01'
end_date = yesterday
# 指数
index_symbol = "000300.XSHG" # 沪深300
# index_symbol = "000905.XSHG" # 中证500
# index_symbol = "000906.XSHG" # 中证800
# index_symbol = "000852.XSHG" # 中证1000
# index_symbol = "000985.XSHG" # 中证全指

### 指数成分股获取

In [4]:
# 获取指数成分股（日期范围查询）
constituents_range = client.get_index_constituents_df(
    index_symbol=index_symbol, 
    start_date=init_date,
    end_date=end_date
)
constituents_range

,datetime,index_symbol,update_date,symbol,weight,display_name
0,2016-01-04,000300.XSHG,2015-12-31,000001.XSHE,0.698,平安银行
1,2016-01-04,000300.XSHG,2015-12-31,000002.XSHE,1.929,万科A
2,2016-01-04,000300.XSHG,2015-12-31,000009.XSHE,0.233,中国宝安
3,2016-01-04,000300.XSHG,2015-12-31,000027.XSHE,0.119,深圳能源
4,2016-01-04,000300.XSHG,2015-12-31,000039.XSHE,0.158,中集集团
...,...,...,...,...,...,...
750595,2026-04-23,000300.XSHG,2026-03-31,688303.XSHG,0.056,大全能源
750596,2026-04-23,000300.XSHG,2026-03-31,688396.XSHG,0.097,华润微
750597,2026-04-23,000300.XSHG,2026-03-31,688472.XSHG,0.078,阿特斯
750598,2026-04-23,000300.XSHG,2026-03-31,688506.XSHG,0.094,百利天恒


In [5]:
# 时间范围内成分股列表
stock_list = constituents_range['symbol'].unique().tolist()

### 行情数据获取

In [6]:
# 获取基准行情
os.environ['ENABLE_TS'] = 'true'
benchmark = client.get_index_daily_df(
    fields="*",
    symbols='000300.SH',
    start_date=init_date,
    end_date=end_date,
    type="index",
)
benchmark

,date,symbol,open,low,pre_close,high,close,change,pct_change,vol,amount
0,2016-01-04,000300.XSHG,3725.8561,3468.9485,3731.0047,3726.2446,3469.0662,-261.9385,-7.0206,115370674.0,1.459682e+08
1,2016-01-05,000300.XSHG,3382.1769,3377.2799,3469.0662,3518.2170,3478.7797,9.7135,0.2800,162116984.0,1.960171e+08
2,2016-01-06,000300.XSHG,3482.4064,3468.4666,3478.7797,3543.7394,3539.8082,61.0285,1.7543,145966144.0,1.609472e+08
3,2016-01-07,000300.XSHG,3481.1499,3284.7374,3539.8082,3481.1499,3294.3839,-245.4243,-6.9333,44102641.0,4.713080e+07
4,2016-01-08,000300.XSHG,3371.8710,3237.9307,3294.3839,3418.8508,3361.5632,67.1793,2.0392,185959451.0,2.034989e+08
...,...,...,...,...,...,...,...,...,...,...,...
2497,2026-04-17,000300.XSHG,4728.9951,4714.4669,4736.6077,4738.3236,4728.6716,-7.9361,-0.1675,182915336.0,5.876227e+08
2498,2026-04-20,000300.XSHG,4727.1661,4720.6853,4728.6716,4763.0427,4757.4419,28.7703,0.6084,221633494.0,6.265684e+08
2499,2026-04-21,000300.XSHG,4750.5149,4722.0074,4757.4419,4776.8258,4767.9970,10.5551,0.2219,205614058.0,5.804080e+08
2500,2026-04-22,000300.XSHG,4751.2794,4750.3502,4767.9970,4801.4465,4799.6270,31.6300,0.6634,204481830.0,6.195057e+08


In [7]:
# 日k数据获取
daily_kline = client.get_kline_df(
            symbol=stock_list,  # 时间范围内成分股列表
            start_date=init_date,
            end_date=end_date,
            frequency="1d",
            adjust_type="post",  # 后复权
            market="cn_stock"
)
daily_kline

,open,high,low,close,pre_close,high_limit,low_limit,avg,volume,datetime,symbol,amount,paused
0,1123.91,1126.72,1051.79,1061.16,1122.97,1235.36,1010.58,1097.68,601648.41,2016-01-04,000001.XSHE,6.603761e+08,0.0
1,2892.93,2892.93,2892.93,2892.93,2892.93,2892.93,2892.93,2892.93,0.00,2016-01-04,000002.XSHE,0.000000e+00,1.0
2,313.85,313.85,280.03,280.29,311.20,342.38,280.03,295.35,531455.56,2016-01-04,000008.XSHE,1.570146e+08,0.0
3,109.47,109.47,98.50,98.50,109.47,120.44,98.50,102.46,11034157.83,2016-01-04,000009.XSHE,1.130235e+09,0.0
4,163.92,164.09,147.70,147.87,164.09,180.48,147.70,156.40,1344845.10,2016-01-04,000027.XSHE,2.103237e+08,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1439841,13.52,13.83,13.22,13.32,13.34,16.01,10.67,13.46,60546160.03,2026-04-23,688472.XSHG,8.148615e+08,0.0
1439842,294.08,294.99,283.68,287.52,295.42,354.50,236.34,288.04,1964929.00,2026-04-23,688506.XSHG,5.659831e+08,0.0
1439843,30.28,30.59,29.83,30.05,30.44,36.53,24.35,30.13,4373757.00,2026-04-23,688561.XSHG,1.317839e+08,0.0
1439844,17.56,17.77,17.13,17.29,17.39,20.86,13.91,17.40,32839134.57,2026-04-23,688599.XSHG,5.713267e+08,0.0


In [8]:
os.environ['ENABLE_TS'] = 'true'

# 获取单只股票的财务指标数据
indicator_data = client.get_finance_indicators(
    symbols=stock_list,
    start_date=init_date,
    end_date=end_date
)
indicator_data.head()

,datetime,symbol,pubDate,statDate,eps,dt_eps,total_revenue_ps,revenue_ps,capital_rese_ps,surplus_rese_ps,...,q_sales_qoq,q_op_yoy,q_op_qoq,q_profit_yoy,q_profit_qoq,q_netprofit_yoy,q_netprofit_qoq,equity_yoy,rd_exp,update_flag
0,2016-01-07,601456.XSHG,2016-01-07,2015-06-30,NaN,NaN,1.1673,1.1673,0.2378,0.1682,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2016-01-07,601456.XSHG,2016-01-07,2015-06-30,NaN,NaN,1.1673,1.1673,0.2378,0.1682,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2016-01-08,603260.XSHG,2016-01-08,2015-06-30,0.3,0.3,2.8886,2.8886,1.8443,0.0175,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22109300.0,1
3,2016-01-08,603260.XSHG,2016-01-08,2015-06-30,0.3,0.3,2.8886,2.8886,1.8443,0.0175,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22109300.0,0
4,2016-01-08,603260.XSHG,2016-01-08,2013-12-31,NaN,NaN,4.6665,4.6665,1.2509,0.0834,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.5339,54891000.0,1


In [ ]:
# # 1分钟数据获取
# minute_data = client.get_kline_df(
#             symbol=stock_list,  # 时间范围内成分股列表
#             start_date=init_date,
#             end_date=end_date,
#             frequency="1m",
#             adjust_type="post",  # 后复权
#             market="cn_stock"
# )
# minute_data

### 基本面数据获取

### 财务数据（聚宽，日频，单季度）

In [9]:
# 是否启用Tushare数据源
os.environ['ENABLE_TS'] = 'false'

# 财务指标 finance_indicator
finance_indicator = client.get_indicator_df(
    symbols=stock_list, start_date=init_date, end_date=end_date,
    fields="symbol,datetime,roe,eps"
)
# 资产负债表 daily_fundamental
daily_balance = client.get_balance_sheet_df(
    symbols=stock_list, start_date=init_date, end_date=end_date,
    fields=("symbol,datetime,equities_parent_company_owners,retained_profit,"
            "salaries_payable,other_payable,total_current_liability,total_assets,"
            "surplus_reserve_fund,constru_in_process,total_owner_equities,account_receivable")
)
# 利润表
daily_income = client.get_income_statement_df(
    symbols=stock_list, start_date=init_date, end_date=end_date,
    fields=("symbol,datetime,net_profit,operating_revenue,total_composite_income,"
            "other_composite_income,ci_parent_company_owners,np_parent_company_owners,"
            "total_operating_cost,operating_tax_surcharges,interest_cost_fin,income_tax_expense")
)
# 现金流量表
daily_cash_flow = client.get_cashflow_statement_df(
    symbols=stock_list, start_date=init_date, end_date=end_date,
    fields=(
        "symbol,datetime,net_operate_cash_flow,cash_and_equivalents_at_end,subtotal_invest_cash_outflow,pubDate,statDate,"
        "fix_intan_other_asset_acqui_cash,subtotal_operate_cash_inflow,goods_sale_and_service_render_cash"
    )
)

# # 基本面衍生因子
# fundamental_factor_data = client.get_factors_df(
#     symbols=stock_list,
#     factor_names=["EBIT", "EBITDA","size","net_asset_per_share","net_profit_ttm","np_parent_company_owners_ttm"],  # 指定因子名称
#     start_date=init_date,
#     end_date=end_date,
#     market="cn_stock"
# )

### 估值、市值/行业数据获取

In [10]:
# 获取多只股票的估值数据（包含市值）
os.environ['ENABLE_TS'] = 'false'
daily_indicators = client.get_valuation_df(
    symbols=stock_list,  # 时间范围内成分股列表
    start_date=init_date,
    end_date=end_date
)
daily_indicators
# 市值取market_cap列

,datetime,symbol,capitalization,circulating_cap,market_cap,circulating_market_cap,turnover_ratio,pe_ratio,pe_ratio_lyr,pb_ratio,ps_ratio,pcf_ratio,pcf_ratio2,dividend_ratio,free_cap,free_market_cap,a_cap,a_market_cap
0,2016-01-04,000001.XSHE,1.430868e+06,1.180405e+06,1621.1730,1337.3994,0.4774,7.4202,8.1869,1.0317,1.8031,0.7471,1.3243,1.2262,595330.5743,674.5095,1.430868e+06,1621.1730
1,2016-01-04,000002.XSHE,1.105052e+06,9.720183e+05,2699.6433,2374.6406,NaN,16.7246,17.1455,3.0358,1.6578,-46.9451,13.6089,2.0461,526268.3382,1285.6736,9.735570e+05,2378.3996
2,2016-01-04,000008.XSHE,2.409433e+05,6.407281e+04,255.6408,67.9813,2.1913,272.5148,3291.0437,9.0748,30.3353,282.6093,-374.2663,0.0000,37795.1991,40.1007,2.409433e+05,255.6408
3,2016-01-04,000009.XSHE,1.592107e+05,1.489669e+05,257.2846,240.7305,4.5146,35.6059,84.8214,5.7568,5.3657,19.6441,306.0294,0.1238,121222.9148,195.8962,1.592107e+05,257.2846
4,2016-01-04,000027.XSHE,3.964492e+05,1.437525e+05,350.4611,127.0772,1.5649,16.8350,17.2297,1.6818,3.0035,25.7057,10.6511,1.5083,107110.4600,94.6856,3.964492e+05,350.4611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1439842,2026-04-23,688472.XSHG,3.643140e+05,1.347654e+05,474.3368,175.4646,4.5962,46.4460,46.4460,2.0197,1.1783,-63.7399,8.7452,0.7093,134765.4391,175.4646,3.643140e+05,474.3368
1439843,2026-04-23,688506.XSHG,4.128738e+04,4.128738e+04,1187.0948,1187.0948,0.4759,-112.9863,-112.9863,17.9211,47.1048,190.3331,-50.9872,0.0000,9031.5984,259.6765,4.128738e+04,1187.0948
1439844,2026-04-23,688561.XSHG,6.822527e+04,6.822527e+04,205.0169,205.0169,0.6411,-16.2398,-16.2398,2.7481,4.6684,43.9947,-1484.1521,0.0000,30261.8794,90.9369,6.822527e+04,205.0169
1439845,2026-04-23,688599.XSHG,2.342568e+05,2.342568e+05,380.1987,380.1987,1.4934,-5.4357,-5.4357,1.7465,0.5651,-8.1071,5.4077,0.0000,138365.9179,224.5679,2.342568e+05,380.1987


In [11]:
# 获取申万一级行业成分股复合查询数据
industry_data = client.get_industry_constituents_composite_df(
    category="sw_l1",
    start_date=init_date,
    end_date=end_date
)
industry_data

MemoryError: cannot allocate memory for array

### 季度频数据获取

In [12]:
def parse_date(date_str: str) -> datetime:
    return datetime.strptime(date_str, "%Y-%m-%d")

def quarter_str(year: int, q: int) -> str:
    return f"{year}q{q}"

def calendar_quarter_from_date(dt: datetime):
    q = (dt.month - 1) // 3 + 1
    return dt.year, q

def reported_quarter_from_end_date(end_date_str: str) -> str:
    dt = parse_date(end_date_str)
    y, m = dt.year, dt.month
    # 报告期映射：确保使用“已完整公布”的最近季度
    if m <= 4:          # 1-4月：上一年Q4
        return quarter_str(y - 1, 4)
    elif m <= 7:        # 5-7月：当年Q1
        return quarter_str(y, 1)
    elif m <= 10:       # 8-10月：当年Q2
        return quarter_str(y, 2)
    else:               # 11-12月：当年Q3
        return quarter_str(y, 3)

def quarter_index(year: int, q: int) -> int:
    return year * 4 + q

def quarters_between(init_date_str: str, stat_quarter_str: str, inclusive: bool = True) -> int:
    init_dt = parse_date(init_date_str)
    sy, sq = map(int, re.findall(r'\d+', stat_quarter_str))  # 解析 "YYYYqQ"
    iy, iq = calendar_quarter_from_date(init_dt)             # 起始用日历季度
    diff = quarter_index(sy, sq) - quarter_index(iy, iq)
    return diff + 1 if inclusive else diff

def last_n_quarters(end_quarter_str: str, count: int):
    sy, sq = map(int, re.findall(r'\d+', end_quarter_str))
    seq = []
    y, q = sy, sq
    for _ in range(count):
        seq.append(quarter_str(y, q))
        q -= 1
        if q == 0:
            y -= 1
            q = 4
    return list(reversed(seq))

### 公告日财务数据（聚宽，季度频，单季度）

In [13]:
# 是否启用Tushare数据源
os.environ['ENABLE_TS'] = 'false'

# 计算 stat_date
stat_date = reported_quarter_from_end_date(end_date)  # -> "2024q4"

# 计算季度差（包含两端 / 不包含两端，按需选择）
quarter_diff_inclusive = quarters_between(init_date, stat_date, inclusive=True)   # 24
# quarter_diff_exclusive = quarters_between(init_date, stat_date, inclusive=False)  # 23

# 资产负债表
balance_statement_data = client.get_history_report_df(
    symbols=stock_list,
    report_sheet="balance",
    fields="year,quarter,symbol,pubDate,statDate,account_receivable,retained_profit,surplus_reserve_fund,total_current_liability,salaries_payable,other_payable,constru_in_process,total_owner_equities",
    count=quarter_diff_inclusive+8,
    stat_date=stat_date
)
# 利润表
income_statement_data = client.get_history_report_df(
    symbols=stock_list,
    report_sheet="income",
    fields="year,quarter,symbol,pubDate,statDate,basic_eps,interest_cost_fin,net_profit,income_tax_expense,financial_expense,total_operating_cost,other_composite_income,np_parent_company_owners",
    count=quarter_diff_inclusive+8,
    stat_date=stat_date
)
# 现金流
cash_flow_statement_data = client.get_history_report_df(
    symbols=stock_list,
    report_sheet="cash_flow",
    fields="year,quarter,symbol,pubDate,statDate,goods_sale_and_service_render_cash,cash_and_equivalents_at_end,fix_intan_other_asset_acqui_cash,subtotal_operate_cash_inflow",
    count=quarter_diff_inclusive+8,
    stat_date=stat_date
)

income_statement_data['EBIT'] = income_statement_data['net_profit'] + income_statement_data['income_tax_expense'] + income_statement_data['financial_expense']

## 三、因子计算

In [ ]:
# 获取多个标的的因子数据
multi_factor_data = client.get_factors_df(
    symbols=stock_list,
    factor_names=["f_njy_001"],  # 单个因子
    start_date=init_date,
    end_date=end_date,
    market="cn_stock"
)
multi_factor_data

,datetime,symbol,factor_name,factor_value
0,2016-01-04,000001.XSHE,f_njy_001,-1.178789
1,2016-01-04,000002.XSHE,f_njy_001,-1.299403
2,2016-01-04,000008.XSHE,f_njy_001,1.897169
3,2016-01-04,000009.XSHE,f_njy_001,4.079571
4,2016-01-04,000027.XSHE,f_njy_001,3.707921


In [15]:
factors_df = multi_factor_data.copy()

## 四、因子评估

### 构建价格数据

In [ ]:
# 多资产价格数据(开盘价买入)
prices_df = daily_kline[daily_kline["datetime"] >= init_date]
prices = prices_df.pivot_table(index="datetime", columns="symbol", values="open")
prices

### 原始因子数据处理

In [17]:
# 对齐数据，转换格式
# 1. symbol + factor_name 排序并 shift
factors_df = factors_df.sort_values(["symbol", "factor_name", "datetime"]).copy()
factors_df["factor_value"] = (
    factors_df.groupby(["symbol", "factor_name"])["factor_value"].shift(1)
)

# 2. 指数成分股历史对齐
factors_df = pd.merge(
    constituents_range[["datetime", "symbol"]],
    factors_df,
    how="left",
    on=["datetime", "symbol"]
)

# 3. 转换成[datetime, symbol]双重索引的factor_table
factors = factors_df.pivot_table(
    index=["datetime", "symbol"],
    columns="factor_name",
    values="factor_value"
)

In [18]:
factors

factor_name             f_njy_001
datetime   symbol                
2016-01-05 000001.XSHE  -1.178789
           000002.XSHE  -1.299403
           000009.XSHE   4.079571
           000027.XSHE   3.707921
           000039.XSHE   1.806191
...                           ...
2026-04-23 688303.XSHG -38.932509
           688396.XSHG   1.210568
           688472.XSHG   3.755368
           688506.XSHG -42.725860
           688981.XSHG  -3.700696

[750253 rows x 1 columns]

### 市值行业数据处理

In [ ]:
# 市值行业转化为series
market_cap_series = (
    daily_indicators
    .set_index(["datetime", "symbol"])["market_cap"]
    .sort_index()
)

industry_series = (
    industry_data
    .set_index(["datetime", "symbol"])["industry_code"]
    .sort_index()
)

benchmark_series = (
    benchmark
    .set_index(["datetime"])["open"]
    .sort_index()
)


In [ ]:
clean_data = fe.get_clean_factor_and_forward_returns(
    factor_data=factors["f_njy_001"],
    prices_data=prices,
    market_cap_data=market_cap_series,
    industry_data=industry_series,
    period=20,
    quantiles=5,
    standardize_when="both",
    max_loss=0.35
)
clean_data

Dropped 0.8% entries from factor data: 0.8% in forward returns computation and 0.0% in binning phase (set max_loss=0 to see potentially suppressed Exceptions).
max_loss is 35.0%, not exceeded: OK!


In [21]:
ic_ts, ic_stats = fe.ic_analysis(clean_data)
ic_stats

{'ic_mean': 0.0424,
 'icir_annualized': 1.8571,
 'ic_win_rate': 0.7069,
 'ic_t_stat': 7.1014,
 'ic_p_value': 0.0,
 'ic_se': 0.005973,
 'ic_test_method': 'newey_west_hac',
 'nw_lags': 19}

In [ ]:
backtest_data, backtest_stats = fe.layered_backtest(clean_data, benchmark_series)
backtest_stats

{'monotonicity': 0.9999999999999999,
 'top_quantile': 5,
 'bottom_quantile': 1,
 'top_cum_return': 1.1962893890687427,
 'top_annual_return': 0.08071028415104942,
 'top_annual_volatility': 0.048762571931406075,
 'top_max_drawdown': -0.39585285169121476,
 'top_sharpe_ratio': 1.61634993403963,
 'top_calmar_ratio': 0.20388961152162552,
 'best_quantile': 5,
 'worst_quantile': 1,
 'best_cum_return': 1.1962893890687427,
 'best_annual_return': 0.08071028415104942,
 'best_annual_volatility': 0.048762571931406075,
 'best_max_drawdown': -0.39585285169121476,
 'best_sharpe_ratio': 1.61634993403963,
 'best_calmar_ratio': 0.20388961152162552,
 'worst_cum_return': 0.036548172039291726,
 'worst_annual_return': 0.003547599883832575,
 'worst_annual_volatility': 0.053692967551851795,
 'worst_max_drawdown': -0.5238515986676298,
 'worst_sharpe_ratio': 0.09274947152109494,
 'worst_calmar_ratio': 0.0067721467164662316,
 'ls_cum_return': 1.1093762528415039,
 'ls_annual_return': 0.0764139651364788,
 'ls_annual

In [24]:
fig = fp.plot_rank_ic_plotly(ic_ts, title="f_njy_001 Rank IC 序列图")
fig.show()

In [ ]:
fig = fp.plot_rank_ic_plotly(ic_ts, title="f_njy_001  Rank IC 序列图", freq="M")
fig.show()

In [25]:
report = fe.evaluate_factor(
    factor_data=factors["f_njy_001"],
    prices_data=prices,
    market_cap_data=market_cap_series,
    industry_data=industry_series,
    period=1,
    quantiles=5,
    standardize_when="both",
    max_loss=0.35
)

Dropped 0.0% entries from factor data: 0.0% in forward returns computation and 0.0% in binning phase (set max_loss=0 to see potentially suppressed Exceptions).
max_loss is 35.0%, not exceeded: OK!


In [79]:
report

ic_mean                               0.033
icir_annualized                      7.2209
ic_win_rate                          0.6844
ic_t_stat                           23.0598
ic_p_value                              0.0
ic_se                              0.001431
ic_test_method             one_sample_ttest
nw_lags                                   0
factor_return                      0.000417
ols_t_stat                        13.398198
ols_p_value                             0.0
ols_se                             0.000031
ols_test_method            one_sample_ttest
ols_|t_stat|>2                     0.197731
ols_r_squared                        0.0047
ols_hit_rate                       0.623987
monotonicity                            1.0
top_quantile                              5
bottom_quantile                           1
top_cum_return                     5.831731
top_annual_return                  0.207341
top_annual_volatility              0.234461
top_max_drawdown                

In [ ]:
fig = fe.plot_layered_backtest_plotly(
    backtest_data,
    title="f_njy_001 分层与多空累计收益"
)
fig.show()